In [1]:
import cv2
import mediapipe as mp
import numpy as np 
import csv
import time

In [10]:


mp_face_mesh = mp.solutions.face_mesh
mp_draw = mp.solutions.drawing_utils
face_mesh = mp_face_mesh.FaceMesh(max_num_faces=1)


    

# Landmark indices
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]
MOUTH = [61, 291, 81, 178, 13, 14, 311, 402]

def euclidean(x1, x2):
    return np.linalg.norm(x1 - x2)

def eye_ratio(eye):
    return (euclidean(eye[1], eye[5]) + euclidean(eye[2], eye[4])) / (2.0 * euclidean(eye[0], eye[3]))

def mouth_ratio(mouth):
    return (euclidean(mouth[2], mouth[6]) + euclidean(mouth[3], mouth[5])) / (2.0 * euclidean(mouth[0], mouth[4]))

 

Video = cv2.VideoCapture(0)
with open("data.csv", "a", newline="") as f:
    writer = csv.writer(f)


    while True:
        suc, frame = Video.read()
        if not suc:
            break

        LABEL = 1
        frame = cv2.flip(frame, 1)
        conv = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(conv)
        

        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:
                h, w, _ = frame.shape
                landmarks = np.array([(int(lmg.x * w), int(lmg.y * h)) for lmg in face_landmarks.landmark])

                left_eye = landmarks[LEFT_EYE]
                right_eye = landmarks[RIGHT_EYE]
                mouth = landmarks[MOUTH]

            
                ear =(eye_ratio(left_eye) + eye_ratio(right_eye)) / 2.0
                mar = mouth_ratio(mouth)
                writer.writerow([ear, mar, LABEL])

                cv2.putText(frame, f"EAR: {ear:.2f}", (30, 40),cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)
                cv2.putText(frame, f"MAR: {mar:.2f}", (30, 80),cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)
                cv2.putText(frame, f"Label: {LABEL}", (30, 120),cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)

                

        cv2.imshow("Drowsiness Feature Extraction", frame)
        if cv2.waitKey(1) & 0xFF==ord('q'):
            break

Video.release()
cv2.destroyAllWindows()
